In [2]:
import pandas as pd

df = pd.read_csv('train.csv')

In [2]:
#Exercise 1: Duplicate Detection and Removal

print(df.shape)
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)
print(df.shape)

(891, 12)
0
(891, 12)


In [3]:
#Exercise 2: Handling Missing Values

print(df.isnull().sum())

age_median = df['Age'].median()
df['Age'].fillna(age_median, inplace=True)

embarked_mode = df['Embarked'].mode()[0]
df['Embarked'].fillna(embarked_mode, inplace=True)

df['Cabin'].fillna('Unknown', inplace=True)

print(df.isnull().sum().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
866


C:\Users\ierri\AppData\Local\Temp\ipykernel_31044\4035700863.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Age'].fillna(age_median, inplace=True)
C:\Users\ierri\AppData\Local\Temp\ipykernel_31044\4035700863.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using

In [4]:
#Exercise 3: Feature Engineering

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Title'] = df['Title'].replace('Ms', 'Miss')

print(df['Title'].value_counts())

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


<>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\ierri\AppData\Local\Temp\ipykernel_31044\452966572.py:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [4]:
#Exercise 4: Outlier Detection and Handling

import numpy as np

print(df['Fare'].describe())

Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

visual_threshold = df['Fare'].quantile(0.98)
df['Fare_capped'] = np.where(df['Fare'] > visual_threshold, visual_threshold, df['Fare'])

df['Fare_log'] = np.log1p(df['Fare'])

print(df['Fare'].max(), df['Fare_capped'].max())

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64
512.3292 211.3375


In [5]:
#Exercise 5: Data Standardization and Normalization

from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

df['Age_scaled'] = scaler_standard.fit_transform(df[['Age']])
df['Fare_scaled'] = scaler_minmax.fit_transform(df[['Fare_capped']])

print(df[['Age_scaled', 'Fare_scaled']].head())

   Age_scaled  Fare_scaled
0   -0.530377     0.034305
1    0.571831     0.337296
2   -0.254825     0.037499
3    0.365167     0.251257
4    0.365167     0.038091


In [6]:
#Exercise 6: Feature Encoding

df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

print(df.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'FamilySize', 'Fare_capped', 'Fare_log',
       'Sex_male', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr',
       'Title_Mrs', 'Title_Rare'],
      dtype='str')


In [7]:
#Exercise 7: Data Transformation for Age Feature

bins = [0, 12, 18, 60, 100]
age_labels = ['Child', 'Teen', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=age_labels)
df = pd.get_dummies(df, columns=['AgeGroup'], prefix='Age')

print(df.filter(like='Age_').head())

   Age_Child  Age_Teen  Age_Adult  Age_Senior
0      False     False       True       False
1      False     False       True       False
2      False     False       True       False
3      False     False       True       False
4      False     False       True       False
